In [1]:
import os
import sys

# 1. Point to your JDK (Make sure you point to a 'jdk' folder, NOT 'jre')
# If you haven't installed JDK 11 yet, download it, then put the correct path here:
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jre1.8.0_461"  

# 2. Point to your Spark folder
os.environ["SPARK_HOME"] = r"C:\spark-3.5.7"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_MASTER_IP"] = "127.0.0.1"
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Initialize findspark with your corrected environment
import findspark
findspark.init(os.environ["SPARK_HOME"])

from pyspark.sql import SparkSession

print("--- Final Environment Check ---")
print("Correct Python Active?:", sys.executable)
print("Using Spark Home:", os.environ.get("SPARK_HOME"))
print("Using Java Home:", os.environ.get("JAVA_HOME"))
print("--------------------------------")



--- Final Environment Check ---
Correct Python Active?: C:\Users\sahil\AppData\Local\Programs\Python\Python311\python.exe
Using Spark Home: C:\spark-3.5.7
Using Java Home: C:\Program Files\Java\jre1.8.0_461
--------------------------------


In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
## Set Pyspark SQL libraries

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [4]:
# Set spark sessions 

spark = SparkSession.builder \
    .appName('pyspark-by-examples') \
    .master('local[4]') \
    .getOrCreate()
    
print("\n🎉 Success! Spark Session Confirmed via 127.0.0.1")


🎉 Success! Spark Session Confirmed via 127.0.0.1


In [5]:
# Simple test data creation
df = spark.read.option('header',True)\
    .option('inferschema', True)\
    .csv("C:/Users/sahil/Downloads/pyspark_test.csv")

In [6]:
# Create a dataFrame

from pyspark.sql import Row
data = [Row(id=1, name='Alice', age=30, salary=70000.0, dept='HR'),
        Row(id=2, name='Bob',   age=25, salary=55000.0, dept='IT'),
        Row(id=3, name='Carol', age=35, salary=90000.0, dept='IT'),
        Row(id=4, name='Dave',  age=28, salary=60000.0, dept='Finance'),
        Row(id=5, name='Eve',   age=None, salary=None,  dept='HR')]
other = spark.createDataFrame(data)
other.show()


+---+-----+----+-------+-------+
| id| name| age| salary|   dept|
+---+-----+----+-------+-------+
|  1|Alice|  30|70000.0|     HR|
|  2|  Bob|  25|55000.0|     IT|
|  3|Carol|  35|90000.0|     IT|
|  4| Dave|  28|60000.0|Finance|
|  5|  Eve|NULL|   NULL|     HR|
+---+-----+----+-------+-------+



In [7]:
# show schema
df.show()
df.printSchema()
df.columns
df.dtypes

+-------+-------+-----+
|  Class|Student|Marks|
+-------+-------+-----+
|class_1|  Alice|   15|
|class_1|    Bob|   89|
|class_1|  Carol|   35|
|class_1|   Farf|    1|
|class_2|    Bob|   18|
|class_2|  Carol|   51|
|class_2|   Dave|   95|
|class_2|    Eve|   49|
|class_2|   Farf|   16|
+-------+-------+-----+

root
 |-- Class: string (nullable = true)
 |-- Student: string (nullable = true)
 |-- Marks: integer (nullable = true)



[('Class', 'string'), ('Student', 'string'), ('Marks', 'int')]

In [62]:
## Select Values from spark dataframes

df.select('Class')
df.select('Class', 'Student').show()

#Rename a single columns
df.select('Class', 'Student')\
    .withColumnRenamed('Class', 'Class_names')\
    .head(n = 1)

#Rename multiple columns
df.select('Class', 'Student')\
    .withColumnRenamed('Class', 'ClassNames')\
    .withColumnRenamed('Student', 'Students')\
    .head(n = 1)

#Rename all columns
df.select('Class', 'Student')\
    .toDF('ClassNames', 'Students')\
    .head(n = 1)

#Other ways
df.select(F.col('Student').alias('Students')).\
    head(n = 1)

+-------+-------+
|  Class|Student|
+-------+-------+
|class_1|  Alice|
|class_1|    Bob|
|class_1|  Carol|
|class_1|   Farf|
|class_2|    Bob|
|class_2|  Carol|
|class_2|   Dave|
|class_2|    Eve|
|class_2|   Farf|
+-------+-------+



[Row(Students='Alice')]

In [92]:
# Manupulate a columns values

df.select((F.col('Marks') * 100).alias('newMarks'))\
    .head(n = 1)

# Select all columns except few
df.select([c for c in df.columns if c != 'Marks']).head(n = 1)

#Select distinct values of a column
df.select('Class').distinct().show()

+-------+
|  Class|
+-------+
|class_1|
|class_2|
+-------+



In [121]:
# Filters in pyspark

df.filter((F.col('Class') == 'class_1') & (F.col('Marks') > 10))\
    .show()

#Filter null values
test = other.withColumnRenamed('salary', 'salaries')
test.filter(F.col('salaries').isNull()).show()

# Filter with expressions
other.filter(F.col('dept').like('%Fina')).show()

# filter with regex
other.filter(F.col('dept').rlike('[Fina]')).show()

# Filter many values
other = other.withColumnRenamed('name', 'names')
other.filter(F.col('names').isin(['Alice', 'Bob'])).show()

+-------+-------+-----+
|  Class|Student|Marks|
+-------+-------+-----+
|class_1|  Alice|   15|
|class_1|    Bob|   89|
|class_1|  Carol|   35|
+-------+-------+-----+

+---+----+----+--------+----+
| id|name| age|salaries|dept|
+---+----+----+--------+----+
|  5| Eve|NULL|    NULL|  HR|
+---+----+----+--------+----+

+---+----+---+------+----+
| id|name|age|salary|dept|
+---+----+---+------+----+
+---+----+---+------+----+

+---+----+---+-------+-------+
| id|name|age| salary|   dept|
+---+----+---+-------+-------+
|  4|Dave| 28|60000.0|Finance|
+---+----+---+-------+-------+

+---+-----+---+-------+----+
| id|names|age| salary|dept|
+---+-----+---+-------+----+
|  1|Alice| 30|70000.0|  HR|
|  2|  Bob| 25|55000.0|  IT|
+---+-----+---+-------+----+



In [99]:
# Sort Data

df.orderBy(F.col('Class').asc(), F.col('Marks').desc()).show()

+-------+-------+-----+
|  Class|Student|Marks|
+-------+-------+-----+
|class_1|    Bob|   89|
|class_1|  Carol|   35|
|class_1|  Alice|   15|
|class_1|   Farf|    1|
|class_2|   Dave|   95|
|class_2|  Carol|   51|
|class_2|    Eve|   49|
|class_2|    Bob|   18|
|class_2|   Farf|   16|
+-------+-------+-----+



In [18]:
new_data = data = [Row(Class= 'Class_3', name='Alice', Marks =30),
        Row(Class= 'Class_3', name='Bob', Marks =40)]
new_df = spark.createDataFrame(new_data)
new_df.show()

+-------+-----+-----+
|  Class| name|Marks|
+-------+-----+-----+
|Class_3|Alice|   30|
|Class_3|  Bob|   40|
+-------+-----+-----+



In [119]:
# Addition of new rows 

df.union(new_df)

# Addition of new columns

# A statis column
df.withColumn('Country', F.lit('Canada'))

# Add a computed columnd
df.withColumn('Percentage', F.col('Marks')/100)

# Add conditional columns
df.withColumn('Grades',
                  F.when(F.col('Marks') > 80, 'A')
                  .when(F.col('Marks') > 40, 'B')
                  .otherwise('F')
             )

# Concat different columns 
df.withColumn('concatenated_values', F.concat(F.col('Student'), F.lit("_"), F.col('Class')))
df.withColumn('concatenated_values', F.concat(F.col('Student'), F.lit("_"), F.col('Marks'))).show()

## lower
df.withColumn('lower', F.lower(F.col("Student"))).show()

+-------+-------+-----+-------------------+
|  Class|Student|Marks|concatenated_values|
+-------+-------+-----+-------------------+
|class_1|  Alice|   15|           Alice_15|
|class_1|    Bob|   89|             Bob_89|
|class_1|  Carol|   35|           Carol_35|
|class_1|   Farf|    1|             Farf_1|
|class_2|    Bob|   18|             Bob_18|
|class_2|  Carol|   51|           Carol_51|
|class_2|   Dave|   95|            Dave_95|
|class_2|    Eve|   49|             Eve_49|
|class_2|   Farf|   16|            Farf_16|
+-------+-------+-----+-------------------+

+-------+-------+-----+-----+
|  Class|Student|Marks|lower|
+-------+-------+-----+-----+
|class_1|  Alice|   15|alice|
|class_1|    Bob|   89|  bob|
|class_1|  Carol|   35|carol|
|class_1|   Farf|    1| farf|
|class_2|    Bob|   18|  bob|
|class_2|  Carol|   51|carol|
|class_2|   Dave|   95| dave|
|class_2|    Eve|   49|  eve|
|class_2|   Farf|   16| farf|
+-------+-------+-----+-----+



In [33]:
# drop data where a column is null

other.dropna(subset = ['salary']).show()
other.dropna().show()

+---+-----+---+-------+-------+
| id| name|age| salary|   dept|
+---+-----+---+-------+-------+
|  1|Alice| 30|70000.0|     HR|
|  2|  Bob| 25|55000.0|     IT|
|  3|Carol| 35|90000.0|     IT|
|  4| Dave| 28|60000.0|Finance|
+---+-----+---+-------+-------+

+---+-----+---+-------+-------+
| id| name|age| salary|   dept|
+---+-----+---+-------+-------+
|  1|Alice| 30|70000.0|     HR|
|  2|  Bob| 25|55000.0|     IT|
|  3|Carol| 35|90000.0|     IT|
|  4| Dave| 28|60000.0|Finance|
+---+-----+---+-------+-------+



In [41]:
# Fill all nulls with a value
df.fillna(0)          # numeric columns → 0
df.fillna('Unknown')  # string columns → 'Unknown'
 
# Fill specific columns with different values
other.fillna({'salary': 50000.0, 'age': 30, 'dept': 'Unknown'})
 
# Fill with column mean
mean_salary = other.select(F.mean('salary')).first()[0]
mean_salary

68750.0

In [46]:
# Update values

other.withColumn(
    'new_id', 
    F.when(F.col('dept') == 'IT', F.col('id') * 4)
    .when(F.col('dept') == 'HR', F.col('id') * 14)
    .otherwise(F.col('id'))
).show()

+---+-----+----+-------+-------+------+
| id| name| age| salary|   dept|new_id|
+---+-----+----+-------+-------+------+
|  1|Alice|  30|70000.0|     HR|    14|
|  2|  Bob|  25|55000.0|     IT|     8|
|  3|Carol|  35|90000.0|     IT|    12|
|  4| Dave|  28|60000.0|Finance|     4|
|  5|  Eve|NULL|   NULL|     HR|    70|
+---+-----+----+-------+-------+------+



In [49]:
# Create a custom definition UDF user defined functions

def create_buckets(age):
    return str(round(age//10,0)*10) + "-" + str(round(age//10,0)*10+10)

# 2. Register it as a Spark UDF (specify the return type!)
create_buckets_udf = F.udf(create_buckets, StringType())

# 3. Apply it to your DataFrame
test = other.filter(F.col('age').isNotNull())
test.withColumn('ages_buckets', create_buckets_udf(F.col('age'))).show()

+---+-----+---+-------+-------+------------+
| id| name|age| salary|   dept|ages_buckets|
+---+-----+---+-------+-------+------------+
|  1|Alice| 30|70000.0|     HR|       30-40|
|  2|  Bob| 25|55000.0|     IT|       20-30|
|  3|Carol| 35|90000.0|     IT|       30-40|
|  4| Dave| 28|60000.0|Finance|       20-30|
+---+-----+---+-------+-------+------------+



In [64]:
# a different type of UDF called pandas UDF which are more efficient and faster
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf(StringType())
def create_buckets_pandas (age: pd.Series) -> pd.Series:
    # Local row-by-row helper function
    def create_age_buckets(age):
        return str(round(age//10,0)*10) + "-" + str(round(age//10,0)*10+10)
    # Use Pandas .apply() to execute the function over the series safely
    return age.apply(create_age_buckets)

In [65]:
# 3. Apply it to your DataFrame
test = other.filter(F.col('age').isNotNull())
test.withColumn('ages_buckets', create_buckets_pandas(F.col('age'))).show()

+---+-----+---+-------+-------+------------+
| id| name|age| salary|   dept|ages_buckets|
+---+-----+---+-------+-------+------------+
|  1|Alice| 30|70000.0|     HR|       30-40|
|  2|  Bob| 25|55000.0|     IT|       20-30|
|  3|Carol| 35|90000.0|     IT|       30-40|
|  4| Dave| 28|60000.0|Finance|       20-30|
+---+-----+---+-------+-------+------------+



In [75]:
# deleting data from data frames

# row deletion is equivalent filter
other.filter(F.col("id") == 1).show()

## columns
other.drop('name', 'dept').show()

+---+-----+---+-------+----+
| id| name|age| salary|dept|
+---+-----+---+-------+----+
|  1|Alice| 30|70000.0|  HR|
+---+-----+---+-------+----+

+---+----+-------+
| id| age| salary|
+---+----+-------+
|  1|  30|70000.0|
|  2|  25|55000.0|
|  3|  35|90000.0|
|  4|  28|60000.0|
|  5|NULL|   NULL|
+---+----+-------+



In [79]:
## select top values

## ascending
other.orderBy(F.col("salary").asc()).limit(4).show()

## descneding
other.orderBy(F.col("salary").desc()).limit(4).show()

+---+-----+----+-------+-------+
| id| name| age| salary|   dept|
+---+-----+----+-------+-------+
|  5|  Eve|NULL|   NULL|     HR|
|  2|  Bob|  25|55000.0|     IT|
|  4| Dave|  28|60000.0|Finance|
|  1|Alice|  30|70000.0|     HR|
+---+-----+----+-------+-------+

+---+-----+---+-------+-------+
| id| name|age| salary|   dept|
+---+-----+---+-------+-------+
|  3|Carol| 35|90000.0|     IT|
|  1|Alice| 30|70000.0|     HR|
|  4| Dave| 28|60000.0|Finance|
|  2|  Bob| 25|55000.0|     IT|
+---+-----+---+-------+-------+



In [98]:
#Aggregate functions in pyspark
##Aggregae functions available -> min, max, count, avg, count_distinct, first, last

test = df.withColumnRenamed('student', 'students')
test.agg(F.sum('Marks').alias('totals'), F.count('students'), F.count_distinct('students'), F.first('students')).show()

# How to fetch values in variables
spark_data = test.agg(F.sum('Marks').alias('totals'), F.count('students'), F.count_distinct('students'), F.first('students')).first()
print("{totals}".format(totals = spark_data[0]))


+------+---------------+------------------------+---------------+
|totals|count(students)|count(DISTINCT students)|first(students)|
+------+---------------+------------------------+---------------+
|   369|              9|                       6|          Alice|
+------+---------------+------------------------+---------------+

369


In [127]:
data = [
    Row(id = 'Alice', Class = 'Class_1'),
    Row(id = 'Bob', Class = 'Class_2'),
    Row(id = 'Carol', Class = 'Class_3'),
    Row(id = 'Dave', Class = 'Class_4')
]
data = spark.createDataFrame(data)
data.show()

+-----+-------+
|   id|  Class|
+-----+-------+
|Alice|Class_1|
|  Bob|Class_2|
|Carol|Class_3|
| Dave|Class_4|
+-----+-------+



In [131]:
## Join formats

other.join(data, F.col('names') == F.col('id'), how = "left").show()

# Joins -> left, right, outer, inner, fill, leftsemi, leftanti
# LEFT SEMI JOIN — like INNER but returns only LEFT columns
# LEFT ANTI JOIN — returns left rows with NO match on right

+---+-----+----+-------+-------+-----+-------+
| id|names| age| salary|   dept|   id|  Class|
+---+-----+----+-------+-------+-----+-------+
|  1|Alice|  30|70000.0|     HR|Alice|Class_1|
|  2|  Bob|  25|55000.0|     IT|  Bob|Class_2|
|  3|Carol|  35|90000.0|     IT|Carol|Class_3|
|  4| Dave|  28|60000.0|Finance| Dave|Class_4|
|  5|  Eve|NULL|   NULL|     HR| NULL|   NULL|
+---+-----+----+-------+-------+-----+-------+



In [145]:
## groupbys

test = df.withColumnRenamed('Student', 'Students')
test.groupby('Class').agg(F.sum(F.col('Marks'))).show()

test.groupby('Class', 'students').agg(F.sum(F.col('Marks'))).show()

test.groupby('Class').agg(F.sum(F.col('Marks')), F.count_distinct(F.col('students'))).show()

+-------+----------+
|  Class|sum(Marks)|
+-------+----------+
|class_1|       140|
|class_2|       229|
+-------+----------+

+-------+--------+----------+
|  Class|students|sum(Marks)|
+-------+--------+----------+
|class_1|   Carol|        35|
|class_1|   Alice|        15|
|class_2|     Bob|        18|
|class_2|   Carol|        51|
|class_2|    Farf|        16|
|class_1|    Farf|         1|
|class_2|    Dave|        95|
|class_2|     Eve|        49|
|class_1|     Bob|        89|
+-------+--------+----------+

+-------+----------+------------------------+
|  Class|sum(Marks)|count(DISTINCT students)|
+-------+----------+------------------------+
|class_1|       140|                       4|
|class_2|       229|                       5|
+-------+----------+------------------------+



In [150]:
## having

test.groupby('Class')\
    .agg(F.sum(F.col('Marks')), F.count_distinct(F.col('students')).alias('numbers'))\
    .filter(F.col('numbers') == 4)\
    .show()

+-------+----------+-------+
|  Class|sum(Marks)|numbers|
+-------+----------+-------+
|class_1|       140|      4|
+-------+----------+-------+



In [151]:
spark.stop()